### Imports

In [ ]:
from pathlib import Path

import pandas as pd

### Read the data from the Excel file

In [ ]:
RAW_PATH = Path("../data/raw")
assert RAW_PATH.exists(), "Create raw folder and add the necessary files inside."

In [ ]:
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(exist_ok=True)

In [ ]:
SIMULADOR_FILE = RAW_PATH / "Simulador Bloco e Subbloco_v2.xlsx"
assert SIMULADOR_FILE.exists(), "Missing Simulador file add to data/raw folder."

In [ ]:
simulador = pd.ExcelFile(SIMULADOR_FILE, engine="calamine")
print(simulador.sheet_names)

In [ ]:
BASES_FILE = RAW_PATH / "Bases_setores_cd.csv"
assert BASES_FILE.exists(), "Missing Bases_setores_cd.csv, add it to data/raw folder."

In [ ]:
bases = pd.ExcelFile(BASES_FILE, engine="calamine")
print(bases.sheet_names)

In [ ]:
DEMAND_FILE = RAW_PATH / "Unifesp_Demanda_v2.csv"
assert DEMAND_FILE.exists(), "Mising Demanda v2 in raw folder."

# Data cleaning steps

1. Extract 'Calendário FV' sheet from the provided Simulador Excel file.
    - extract sheet by sheet name
    - drop empty columns
    - drop index column
2. Extract 'Base Demanda' sheet from the provided Simulador Excel file.
    - extract sheet by sheet name
    - filter columns (not calculated columns)
    - merge with 'Calendário FV' sheet to get the 'Dia Captacao' column
3. Extract 'Setor vs Zoneamento' sheet from the provided Bases setores cd Excel file.
    - extract sheet by sheet name
3. Extract 'GI' sheet from the provided Bases setores cd Excel file.
    - extract sheet by sheet name

### Extract calendar data from the 'Calendário FV' sheet

In [ ]:
df_calendario = simulador.parse("Calendário FV")

In [ ]:
# drop empty columns
df_calendario = df_calendario.dropna(how="all", axis=1)

In [ ]:
df_calendario = df_calendario.drop(columns=["#"])

In [ ]:
df_calendario.columns

In [ ]:
# export csv
output_path = PROCESSED_PATH / "calendario_fv.csv"
df_calendario.to_csv(output_path, index=False)

### Extract demand from  'Base Demanda' sheet

Notes:
- Dia Captacao depends on data from calendar that seems to not be present in the 'Calendário FV' sheet. It is possible that this data is in another sheet or file. Further investigation is needed to locate the source of this information. Or even if this is needed. Initial thought it that it's not needed.

In [ ]:
df_demand = simulador.parse("Base Demanda")

In [ ]:
DEMAND_COLUMNS_NEEDED = [
    "data_pedido",
    "cd_setor",
    "cd_cd",
    "nm_ciclo",
    "aa_ciclo",
    "total_pedidos",
    "total_volumes",
    "total_itens",
]
df_demand = df_demand[DEMAND_COLUMNS_NEEDED]

In [ ]:
df_demand["ciclo"] = df_demand["aa_ciclo"].astype(str) + df_demand["nm_ciclo"].astype(
    str
).str.zfill(2)

In [ ]:
df_calendario["CICLOS"] = df_calendario["CICLOS"].astype(str)

In [ ]:
n_before = len(df_demand)
df = df_demand.merge(
    df_calendario,
    left_on=["ciclo", "cd_setor"],
    right_on=["CICLOS", "COD SETOR"],
    how="left",
    indicator=True,
)
unmatched = df["_merge"].eq("left_only").sum()
print(f"{unmatched}/{n_before} demand rows unmatched to calendar ({unmatched / n_before:.1%})")

In [ ]:
df = df_demand.merge(df_calendario, left_on=["ciclo", "cd_setor"], right_on=["CICLOS", "COD SETOR"])

In [ ]:
output_file = PROCESSED_PATH / "demanda_with_calendar.csv"
df.to_csv(output_file, index=False)

### Extract the time series for the level forecasting

We need 1 point per (sector, cycle) combination with date = open date of the cycle. This will be used to forecast the level of demand for each sector and cycle. And, we need to aggregate the demand for each sector and cycle combination. The aggregation will be done by summing the demand for each sector and cycle combination. We need to include cycle duration in the time series. The cycle duration is the number of days between the open date and the close date of the cycle. The cycle duration will be used to forecast the level of demand for each sector and cycle.

In [ ]:
df["Dt Abertura"] = pd.to_datetime(df["Dt Abertura"])
df["Dt Fechamento"] = pd.to_datetime(df["Dt Fechamento"])

df_level = (
    df.groupby(["cd_setor", "ciclo"], as_index=False)
    .agg(
        date=("Dt Abertura", "first"),
        cycle_duration=("Qtde dias", "first"),
        total_pedidos=("total_pedidos", "sum"),
        total_volumes=("total_volumes", "sum"),
        total_itens=("total_itens", "sum"),
    )
    .sort_values(by=["cd_setor", "date"])
    .reset_index(drop=True)
)
df_level.head()

In [ ]:
output_file = PROCESSED_PATH / "demanda_level.csv"
df_level.to_csv(output_file, index=False)

### Extract the data for shape forecasting

We need one data point per (sector, cycle, position relative to the cycle start date normalized to the cycle duration) combination. This will be used to forecast the shape of the demand for each sector and cycle. The position relative to the cycle start date normalized to the cycle duration will be used to forecast the shape of the demand for each sector and cycle. Then, we need to show orders for each position as a share of the total orders for the cycle. This will be used to forecast the shape of the demand for each sector and cycle. The share of orders for each position will be used to forecast the shape of the demand for each sector and cycle. 

In terms of how many data points, we can use one per day with order, then we need to transform the date to position relative to the cycle start date normalized to the cycle duration. Then, we need to show orders for each position as a share of the total orders for the cycle.

So first, we need to extract the cycle totals, then we need to extract the position relative to the cycle start date normalized to the cycle duration, then we need to show orders for each position as a share of the total orders for the cycle. 

Since we don't really know whether items, orders or volume is the best measure of demand, we can extract all three and then decide which one to use for the shape forecasting.

In [ ]:
df_shape = df.copy()
# df_shape["cycle_total"] =
df_shape.columns

In [ ]:
df_shape["data_pedido"] = pd.to_datetime(df_shape["data_pedido"])
df_shape["Dt Abertura"] = pd.to_datetime(df_shape["Dt Abertura"])

In [ ]:
df_shape["relative_date"] = (df_shape["data_pedido"] - df_shape["Dt Abertura"]).dt.days / df[
    "Qtde dias"
]

In [ ]:
# exclude orders outside of cycle (don't know if that's the best way to deal with it)
df_shape = df_shape[df_shape["relative_date"] <= 1]

In [ ]:
# daily aggregation: collapses multiple distribution centers / records on the same day for a sector
# into single daily totals
df_shape = df_shape.groupby(
    ["cd_setor", "ciclo", "data_pedido", "relative_date"],
    as_index=False,
)[["total_pedidos", "total_volumes", "total_itens"]].sum()

# extract cycle totals (items, orders, volume)
df_shape[["ciclo_total_pedidos", "ciclo_total_volumes", "ciclo_total_itens"]] = df_shape.groupby(
    ["cd_setor", "ciclo"]
)[["total_pedidos", "total_volumes", "total_itens"]].transform("sum")

# share conversion: convert per day (items, orders, volume) to share of cycle total per sector
df_shape["share_pedidos"] = df_shape["total_pedidos"] / df_shape["ciclo_total_pedidos"]
df_shape["share_volumes"] = df_shape["total_volumes"] / df_shape["ciclo_total_volumes"]
df_shape["share_itens"] = df_shape["total_itens"] / df_shape["ciclo_total_itens"]

df_shape.head()

In [ ]:
output_file = PROCESSED_PATH / "demanda_shape.csv"
df_shape.to_csv(output_file, index=False)

### Extract 'Setor vs Zoneamento' sheet from the provided Bases setores cd Excel file.

In [ ]:
zones = bases.parse("Setor vs Zoneamento")

In [ ]:
zones.head()

### Extract 'GI' sheet from the provided Bases setores cd Excel file.

In [ ]:
gi = bases.parse("GI")

In [ ]:
gi.head()

### Second Demand Sheet

In [ ]:
demand_v2_df = pd.read_csv(
    DEMAND_FILE,
    parse_dates=["data_pedido"],
    dtype={
        "cd_cd": str,
        "cd_setor": str,
        "rota": str,
    },
)

In [ ]:
demand_v2_df.head()

# Exploration

In [ ]:
# I expect that every cycle start and end at the same day if the sector is in
# the same block + sublock
# this is the code to check this hypothesis is correct
is_consistent = (
    df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[["Dt Abertura", "Dt Fechamento"]]
    .nunique()
    .eq(1)
    .all()
    .all()
)

print(f"Expectation holds: {is_consistent}")

In [ ]:
# which blocks/subblocks/cycles diverge (and which sectors differ):
# Count unique start and end dates per block + subblock + cycle
cycle_dates_summary = df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[
    ["Dt Abertura", "Dt Fechamento"]
].nunique()

# Filter for groups that have more than 1 distinct date
discrepancies = cycle_dates_summary[
    (cycle_dates_summary["Dt Abertura"] > 1) | (cycle_dates_summary["Dt Fechamento"] > 1)
]

if discrepancies.empty:
    print(
        "Confirmed: Every cycle starts and ends on the same day for "
        "all sectors in the same block + sublock."
    )
else:
    print(
        f"Found {len(discrepancies)} (BLOCO, SUB BLOCO, CICLOS) combination(s) "
        "with diverging dates:\n"
    )
    display(discrepancies)

    # Inspect the exact sectors and dates causing the mismatch
    conflicting_rows = df_calendario.merge(
        discrepancies.reset_index()[["BLOCO", "SUB BLOCO", "CICLOS"]],
        on=["BLOCO", "SUB BLOCO", "CICLOS"],
    )
    display(
        conflicting_rows[
            [
                "BLOCO",
                "SUB BLOCO",
                "CICLOS",
                "COD SETOR",
                "Dt Abertura",
                "Dt Fechamento",
                "Qtde dias",
            ]
        ].sort_values(
            by=[
                "CICLOS",
                "BLOCO",
                "SUB BLOCO",
                "Qtde dias",
                "COD SETOR",
            ]
        )
    )

In [ ]:
# Qtde dias seems to be cycle length including start and end day
(
    (df_calendario["Dt Fechamento"] - df_calendario["Dt Abertura"]).dt.days + 1
    != df_calendario["Qtde dias"]
).sum()

In [ ]:
# many orders are done outside of cycle prescribed length
# right now we're just dropping those outside of the cycle,
# is this the best way to handle this?

In [ ]:
# I assume that each sector can only be part of one block / sublock per cycle
dupe_check = df_calendario.groupby(["COD SETOR", "CICLOS"])["Dt Abertura"].nunique()
assert (dupe_check <= 1).all(), "Multiple distinct open dates per (setor, ciclo)"

In [ ]:
# check how many rotas each city has
# I imagine that SP has many, but others have less like North
# or Northeast
demand_v2_df.groupby("estado")["rota"].nunique()

In [ ]:
route_counts = demand_v2_df.groupby("cidade")["rota"].nunique()

# Filter to cities with more than 1 route
mult_route_cities = route_counts[route_counts > 1]

mult_route_cities.index.tolist()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Count unique cities per route
cities_per_route = demand_v2_df.groupby("rota")["cidade"].nunique()

# 2. Plot setup
plt.figure(figsize=(9, 5))
sns.set_theme(style="whitegrid")

# 3. Density plot (Histogram + KDE curve)
ax = sns.histplot(
    cities_per_route,
    kde=True,
    stat="density",  # Normalizes height to show probability density
    discrete=True,  # Aligns bars cleanly to integer counts
    color="#2b5c8f",
    edgecolor="white",
    alpha=0.6,
)

# 4. Optional: Add median and mean reference lines
median_val = cities_per_route.median()
mean_val = cities_per_route.mean()

plt.axvline(
    median_val,
    color="darkorange",
    linestyle="--",
    linewidth=1.5,
    label=f"Median: {median_val:.1f}",
)
plt.axvline(
    mean_val,
    color="crimson",
    linestyle=":",
    linewidth=1.5,
    label=f"Mean: {mean_val:.1f}",
)

# 5. Labels and styling
plt.title("Density Distribution of Cities per Route", fontsize=14, pad=12)
plt.xlabel("Number of Cities per Route", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# 1. Count unique routes per city
routes_per_city = demand_v2_df.groupby("cidade")["rota"].nunique()

# 2. Plot setup
plt.figure(figsize=(9, 5))
sns.set_theme(style="whitegrid")

# 3. Histogram with probability (sums to 100%)
ax = sns.histplot(
    routes_per_city,
    stat="probability",  # Bar height = share of cities (e.g., 0.90 = 90%)
    discrete=True,  # Bins cleanly on integers 1, 2, 3...
    color="#2b5c8f",
    edgecolor="white",
    alpha=0.8,
)

# 4. Format X-axis to show only integers 1, 2, 3... N
max_routes = routes_per_city.max()
plt.xticks(range(1, max_routes + 1))

# 5. Format Y-axis as percentage (0% to 100%)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# 6. Add exact percentage labels on top of each bar
for patch in ax.patches:
    height = patch.get_height()
    if height > 0:
        ax.annotate(
            f"{height * 100:.1f}%",
            xy=(patch.get_x() + patch.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

# 7. Labels and styling
plt.title("Proportion of Cities by Number of Routes", fontsize=14, pad=12)
plt.xlabel("Number of Routes (N)", fontsize=11)
plt.ylabel("Percentage of Cities", fontsize=11)

# Leave space at the top for labels
plt.ylim(0, max(p.get_height() for p in ax.patches) * 1.15)
plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns

# 1. Count unique routes per city
routes_per_city = demand_v2_df.groupby("cidade")["rota"].nunique()

# 2. Define buckets: '1' through '10', plus '>10'
categories = [str(i) for i in range(1, 11)] + [">10"]
binned_routes = routes_per_city.apply(lambda x: str(x) if x <= 10 else ">10")

# 3. Calculate proportion for each bucket (ensures all 1-10 are present in order)
pct_per_bucket = binned_routes.value_counts(normalize=True).reindex(categories, fill_value=0.0)

# 4. Plot setup
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    x=pct_per_bucket.index,
    y=pct_per_bucket.values,
    color="#2b5c8f",
    edgecolor="white",
    alpha=0.85,
)

# 5. Format Y-axis as percentage
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# 6. Annotate percentage on top of each bar
for patch in ax.patches:
    height = patch.get_height()
    if height > 0:
        ax.annotate(
            f"{height * 100:.1f}%",
            xy=(patch.get_x() + patch.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

# 7. Labels and styling
plt.title("Proportion of Cities by Number of Routes", fontsize=14, pad=12)
plt.xlabel("Number of Routes", fontsize=11)
plt.ylabel("Percentage of Cities", fontsize=11)

# Leave space at the top for labels
plt.ylim(0, max(pct_per_bucket.values) * 1.15)
plt.tight_layout()

plt.show()

In [ ]:
# Top 15 cities with the highest route count
top_cities = (
    demand_v2_df.groupby("cidade")["rota"].nunique().nlargest(15).sort_values(ascending=True)
)

plt.figure(figsize=(9, 6))
sns.set_theme(style="whitegrid")

ax = top_cities.plot(kind="barh", color="#2b5c8f", edgecolor="white", alpha=0.85)

# Label bars with their exact counts
for bar in ax.patches:
    ax.text(
        bar.get_width() + 0.1,
        bar.get_y() + bar.get_height() / 2,
        f"{int(bar.get_width())}",
        va="center",
        fontsize=10,
    )

plt.title("Top 15 Cities by Number of Distinct Routes", fontsize=14, pad=12)
plt.xlabel("Number of Routes", fontsize=11)
plt.ylabel("City", fontsize=11)
plt.tight_layout()

plt.show()

In [ ]:
demand_v2_df.head()

# States per Sector (cd_setor)

Analyze how many distinct states each sector operates across.

In [ ]:
# 1. Summary statistics of states per sector
states_per_sector = demand_v2_df.groupby("cd_setor")["estado"].nunique()

print(f"Total unique sectors: {len(states_per_sector)}")
print(f"Max states for a single sector: {states_per_sector.max()}")
print(f"Min states: {states_per_sector.min()}")
print(f"Average states per sector: {states_per_sector.mean():.2f}")
print(f"Median states per sector: {states_per_sector.median():.1f}")

print("Frequency of states per sector:")
counts = states_per_sector.value_counts().sort_index()
pcts = states_per_sector.value_counts(normalize=True).sort_index() * 100
summary_df = pd.DataFrame({"count": counts, "percentage": pcts.map("{:.1f}%".format)})
summary_df.T

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# States with the most sectors (cd_setor per estado)
sectors_per_state = (
    demand_v2_df.groupby("estado")["cd_setor"]
    .nunique()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 5))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    x=sectors_per_state.index,
    y=sectors_per_state.values,
    color="#2b5c8f",
    edgecolor="white",
    alpha=0.85,
)

# Annotate count on top of each bar
for patch in ax.patches:
    height = patch.get_height()
    if height > 0:
        ax.annotate(
            f"{int(height)}",
            xy=(patch.get_x() + patch.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
        )

plt.title("Number of Distinct Sectors per State (cd_setor per Estado)", fontsize=14, pad=12)
plt.xlabel("State (Estado)", fontsize=11)
plt.ylabel("Number of Sectors", fontsize=11)
plt.ylim(0, max(sectors_per_state.values) * 1.1)
plt.tight_layout()

plt.show()


In [ ]:
# Density distribution of states per sector (Histogram + KDE)
plt.figure(figsize=(9, 5))
sns.set_theme(style="whitegrid")

ax = sns.histplot(
    states_per_sector,
    kde=True,
    stat="density",
    discrete=True,
    color="#2b5c8f",
    edgecolor="white",
    alpha=0.6,
)

median_val = states_per_sector.median()
mean_val = states_per_sector.mean()

plt.axvline(
    median_val,
    color="darkorange",
    linestyle="--",
    linewidth=1.5,
    label=f"Median: {median_val:.1f}",
)
plt.axvline(
    mean_val,
    color="crimson",
    linestyle=":",
    linewidth=1.5,
    label=f"Mean: {mean_val:.1f}",
)

plt.title("Density Distribution of States per Sector", fontsize=14, pad=12)
plt.xlabel("Number of States per Sector", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 sectors spanning the highest number of distinct states
top_sectors = (
    demand_v2_df.groupby("cd_setor")["estado"].nunique().nlargest(15).sort_values(ascending=True)
)

plt.figure(figsize=(9, 6))
sns.set_theme(style="whitegrid")

ax = top_sectors.plot(kind="barh", color="#2b5c8f", edgecolor="white", alpha=0.85)

for bar in ax.patches:
    ax.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{int(bar.get_width())}",
        va="center",
        fontsize=10,
    )

plt.title("Top 15 Sectors by Number of Distinct States", fontsize=14, pad=12)
plt.xlabel("Number of States", fontsize=11)
plt.ylabel("Sector (cd_setor)", fontsize=11)
plt.tight_layout()
plt.show()